# SymPy verifications of the main theorems from Section 4

Theorem, algorithm and equation numbers are those of the manuscript version of 22 November 2026. The next cell setups SymPy.

In [1]:
import sympy as sp

def check(expr):
    return sp.simplify(expr)

## 1 Verification of Theorem 4.3

### 1.1 Statement and proof (as in the manuscript)

The lower model $l_k$ is a convex combination of the quadratic lower bounds (4.1), $\ell_i(x)=f(y_i)+\langle\nabla f(y_i),\,x-y_i\rangle+\tfrac m2\|x-y_i\|^2$. By (4.4), it is a quadratic with curvature $m$, with minimizer $z_k$ and minimal value $v_k=l_k(z_k)$. The theorem is about one iteration of the following method.

**Algorithm 4.2. (Cheap model-based method, $m>0$)** With the increment $a_k\geq0$, the algorithm applies the following update. Given the minimizer of an upper model $x_k$ and a lower model $l_k\leq f$ with weight $A_k>0$, compute $y_k$ as a solution to (4.8),
$$
\langle\nabla f(y_k),\,A_k(x_k-y_k)+\tau_kA_k\,(z_k-y_k)\rangle\geq0 ,
$$
with $z_k=\mathrm{argmin}\;l_k$ and $\tau_k=a_k/A_{k+1}$, and output a new upper model minimizer $x_{k+1}=y_k-\tfrac1L\nabla f(y_k)$ and a new lower model $l_{k+1}=(A_kl_k+a_k\ell_k)/A_{k+1}$, where $\ell_k$ is given by (4.1) at $y_k$, with the new weight $A_{k+1}=A_k+a_k$. By (4.5), the minimizer of the new lower model is
$$
z_{k+1}=(1-\tau_k)\,z_k+\tau_k\big(y_k-\tfrac1m\nabla f(y_k)\big).
$$

**Theorem 4.3. (Cheap model-based method, $m>0$)** Let $f\in\mathcal{F}_{m,L}$ with $m>0$. At step $k\geq1$, given $x_k$ and $l_k$, let $x_{k+1}$ and $l_{k+1}$ be generated by the cheap model-based method (Algorithm 4.2). Let $\Psi_k=A_k\big(f(x_k)-l_k(z_k)\big)$ be the gap in (4.3) with weights $A_k>0$ and increments $A_{k+1}=A_k+a_k$ with $a_k\geq0$. If $a_k^2\leq qA_{k+1}^2$, then
$$
\Psi_{k+1}\leqslant\Psi_k .
$$

*Proof.* The proof consists of a weighted sum of three inequalities:

- convexity (2.1) between $x_k$ and $y_k$, with weight $\lambda_1=A_k$:

  $$
  f(x_k)\geq f(y_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle,
  $$

- the descent lemma (2.3), with weight $\lambda_2=A_{k+1}$:

  $$
  f(x_{k+1})\leq f(y_k)-\tfrac{1}{2L}\|\nabla f(y_k)\|^2,
  $$

- the relation (4.8), with weight $\lambda_3=1$:

  $$
  \langle\nabla f(y_k),\,A_k(x_k-y_k)+\tau_kA_k\,(z_k-y_k)\rangle\geq0.
  $$

The weighted sum reads

$$
\begin{aligned}
0\geq{}&\lambda_1\big[f(y_k)-f(x_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle\big]\\
&+\lambda_2\big[f(x_{k+1})-f(y_k)+\tfrac{1}{2L}\|\nabla f(y_k)\|^2\big]\\
&+\lambda_3\langle\nabla f(y_k),\,A_k(y_k-x_k)+\tau_kA_k\,(y_k-z_k)\rangle ,
\end{aligned}
$$

which, using the update (4.5) and $A_{k+1}=A_k+a_k$, together with

$$
\begin{aligned}
A_{k+1}l_{k+1}(z_{k+1})&=A_kl_k(z_{k+1})+a_k\ell_k(z_{k+1}),\\
A_kl_k(z_{k+1})&=A_kl_k(z_k)+\tfrac{mA_k}{2}\|z_{k+1}-z_k\|^2,
\end{aligned}
$$

as $A_kl_k$ is a quadratic with curvature $mA_k$ minimized at $z_k$, can be reformulated as

$$
0\geq\Psi_{k+1}-\Psi_k+\tfrac{qA_{k+1}^2-a_k^2}{2mA_{k+1}}\|\nabla f(y_k)\|^2+\tau_kA_k\,\tfrac m2\|y_k-z_k\|^2 .
$$

Both coefficients are nonnegative under our assumption, concluding the proof. $\square$

### 1.2 SymPy verification

To verify the expression symbolically, one can define the following symbols. The minimal value $A_kl_k(z_k)$ of the current lower model is the free symbol `Alk_zk`, and the model is evaluated at $z_{k+1}$ through its quadratic expansion around $z_k$:

In [2]:
L = sp.symbols('L')
m = sp.symbols('m')
Ak = sp.symbols('A_k')
ak = sp.symbols('a_k')

xk = sp.symbols('x_k')
zk = sp.symbols('z_k')
yk = sp.symbols('y_k')
gyk = sp.symbols('\\nabla{}f(y_k)')
fyk = sp.symbols('f(y_k)')
fxk = sp.symbols('f(x_k)')
fxkp1 = sp.symbols('f(x_{k+1})')
Alk_zk = sp.symbols('A_k{}l_k(z_k)')

q = m / L
Akp1 = Ak + ak
tauk = ak / Akp1
zkp1 = (1 - tauk) * zk + tauk * (yk - gyk / m)                  # update (4.5)

ellk_zkp1 = fyk + gyk * (zkp1 - yk) + m / 2 * (zkp1 - yk) ** 2  # ell_k(z_{k+1}), piece (4.1) at y_k
Alk_zkp1 = Alk_zk + m * Ak / 2 * (zkp1 - zk) ** 2               # A_k l_k(z_{k+1}): curvature m A_k, minimized at z_k
Alkp1_zkp1 = Alk_zkp1 + ak * ellk_zkp1                          # A_{k+1} l_{k+1}(z_{k+1})

Now, one can write the three inequalities used in the proof (in the form "$\ldots\leqslant 0$") and their weights to form the weighted sum:

In [3]:
inequality1 = fyk - fxk + gyk * (xk - yk)
inequality2 = fxkp1 - fyk + 1 / 2 / L * gyk ** 2
inequality3 = gyk * (Ak * (yk - xk) + tauk * Ak * (yk - zk))
lambda1 = Ak
lambda2 = Akp1
lambda3 = 1

weighted_sum = lambda1 * inequality1 + lambda2 * inequality2 + lambda3 * inequality3

Similarly, one can define the two values of the gap and form the target inequality

In [4]:
potentialk = Ak * fxk - Alk_zk
potentialkp1 = Akp1 * fxkp1 - Alkp1_zkp1

target_inequality = potentialkp1 - potentialk + (q * Akp1 ** 2 - ak ** 2) / 2 / m / Akp1 * gyk ** 2 + tauk * Ak * m / 2 * (yk - zk) ** 2

It remains to check that the expressions match, which we check by verifying their difference is identically zero

In [5]:
check(weighted_sum - target_inequality)

0

As complements, in the order of the checks below:

- the point $z_{k+1}$ of (4.5) minimizes $A_{k+1}l_{k+1}$, and the minimal value of the averaged model is the one displayed in (4.5),
$$
v_{k+1}=(1-\tau_k)\,v_k+\tau_k\Big(f(y_k)-\tfrac1{2m}\|\nabla f(y_k)\|^2\Big)+\tfrac m2\tau_k(1-\tau_k)\Big\|z_k-y_k+\tfrac1m\nabla f(y_k)\Big\|^2,
$$
where $v_k=l_k(z_k)$ and $v_{k+1}=l_{k+1}(z_{k+1})$;
- the formulas (4.4) for the minimizer $z_k$ and the minimal value $v_k$ of a convex combination of pieces, checked with three pieces whose weights sum to one;
- the largest admissible increment $a_k=\sqrt q\,A_{k+1}$, that is, $A_{k+1}=A_k/(1-\sqrt q)$, gives the constant parameters $\tau_k=\sqrt q$ of the method (2.27): $y_k=x_k+\tfrac{\sqrt q}{1+\sqrt q}(z_k-x_k)$ (the relation (4.8) with equality) and $z_{k+1}=(1-\sqrt q)z_k+\sqrt q\,(y_k-\tfrac1m\nabla f(y_k))$;
- the initial gap used before Corollary 4.4: the first lower model $l_1=\ell_0$ is minimized at $z_1=y_0-\tfrac1m\nabla f(y_0)$, and $\Psi_1=A_1\big(f(x_1)-f(y_0)+\tfrac1{2m}\|\nabla f(y_0)\|^2\big)$.

In [6]:
z = sp.symbols('z')
Alkp1_z = Alk_zk + m * Ak / 2 * (z - zk) ** 2 + ak * (fyk + gyk * (z - yk) + m / 2 * (z - yk) ** 2)   # A_{k+1} l_{k+1}(z)
vk = Alk_zk / Ak
vkp1 = (1 - tauk) * vk + tauk * (fyk - gyk ** 2 / 2 / m) + m / 2 * tauk * (1 - tauk) * (zk - yk + gyk / m) ** 2

# three pieces (y_i, grad f(y_i), f(y_i)) with weights lambda_0, lambda_1 and 1 - lambda_0 - lambda_1
y0, y1, y2 = sp.symbols('y_0 y_1 y_2')
gy0, gy1, gy2 = sp.symbols('\\nabla{}f(y_0) \\nabla{}f(y_1) \\nabla{}f(y_2)')
fy0, fy1, fy2 = sp.symbols('f(y_0) f(y_1) f(y_2)')
lam0, lam1 = sp.symbols('lambda_0 lambda_1')
pieces = [(lam0, y0, gy0, fy0), (lam1, y1, gy1, fy1), (1 - lam0 - lam1, y2, gy2, fy2)]
l_z = sum(lam * (f + g * (z - y) + m / 2 * (z - y) ** 2) for (lam, y, g, f) in pieces)
z_44 = sum(lam * (y - g / m) for (lam, y, g, f) in pieces)                                           # z_k of (4.4)
v_44 = sum(lam * (f - g ** 2 / 2 / m + m / 2 * (y - g / m - z_44) ** 2) for (lam, y, g, f) in pieces)  # v_k of (4.4)

sq = sp.sqrt(q)
ak_const = sq * Ak / (1 - sq)                                # a_k = sqrt(q) A_{k+1} with A_{k+1} = A_k / (1 - sqrt(q))
tau_const = tauk.subs(ak, ak_const)
yk_average = (xk + tau_const * zk) / (1 + tau_const)         # the relation (4.8) with equality

fx1, A1 = sp.symbols('f(x_1) A_1')
l1_z = fy0 + gy0 * (z - y0) + m / 2 * (z - y0) ** 2          # l_1 = ell_0
z1 = y0 - gy0 / m

(check(sp.diff(Alkp1_z, z).subs(z, zkp1)),
 check(Alkp1_zkp1 / Akp1 - vkp1),
 check(l_z - (v_44 + m / 2 * (z - z_44) ** 2)),
 check((ak ** 2 - q * Akp1 ** 2).subs(ak, ak_const)),
 check(tau_const - sq),
 check(yk_average - (xk + sq / (1 + sq) * (zk - xk))),
 check(zkp1.subs(ak, ak_const) - ((1 - sq) * zk + sq * (yk - gyk / m))),
 check(sp.diff(l1_z, z).subs(z, z1)),
 check(A1 * (fx1 - l1_z.subs(z, z1)) - A1 * (fx1 - fy0 + gy0 ** 2 / 2 / m)))

(0, 0, 0, 0, 0, 0, 0, 0, 0)

## 2 Verification of Theorem 4.7

### 2.1 Statement and proof (as in the manuscript)

Here $m=0$, and the lower model is $l_k=\tfrac{1}{A_k}\sum_{i=0}^{k-1}a_i\,\ell_i$ with the affine pieces $\ell_i(x)=f(y_i)+\langle\nabla f(y_i),\,x-y_i\rangle$ and $A_k=\sum_{i=0}^{k-1}a_i$. Its regularized version (4.9), written in (4.14), is
$$
r_k(x)=l_k(x)+\tfrac{L}{2A_k}\|x-x_0\|^2 ,
$$
so that $A_kr_k$ is a quadratic with curvature $L$, minimized at $z_k=x_0-\tfrac1L\sum_{i=0}^{k-1}a_i\nabla f(y_i)$. The theorem is about one iteration of the following method.

**Algorithm 4.6. (Cheap regularized model-based method, $m=0$)** With the increment $a_k\geq0$, the algorithm applies the following update. Given the minimizer of an upper model $x_k$ and a lower model $l_k\leq f$ with weight $A_k>0$, compute $y_k$ as a solution to (4.15),
$$
\langle\nabla f(y_k),\,A_kx_k+a_kz_k-A_{k+1}y_k\rangle\geq0,
$$
with $z_k=\mathrm{argmin}\;r_k$, and output a new upper model minimizer $x_{k+1}=y_k-\tfrac1L\nabla f(y_k)$ and a new lower model $l_{k+1}=(A_kl_k+a_k\ell_k)/A_{k+1}$, where $\ell_k$ is given by (4.1) at $y_k$, with the new weight $A_{k+1}=A_k+a_k$.

**Theorem 4.7. (Cheap regularized model-based method, $m=0$)** Let $f\in\mathcal{F}_{0,L}$. At step $k\geq1$, given $x_k$ and $l_k$, let $x_{k+1}$ and $l_{k+1}$ be generated by the cheap regularized model-based method (Algorithm 4.6). Let $\Psi_k=A_k\big(f(x_k)-r_k(z_k)\big)$ be the gap in (4.12) with weights $A_k>0$ and increments $A_{k+1}=A_k+a_k$ with $a_k\geq0$. If $a_k^2\leq A_{k+1}$, then
$$
\Psi_{k+1}\leqslant\Psi_k .
$$

*Proof.* The proof consists of a weighted sum of the following inequalities, which are three of the four inequalities of the proof of Theorem 2.5:

- convexity (2.1) between $x_k$ and $y_k$, with weight $\lambda_1=A_k$:

  $$
  f(x_k)\geq f(y_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle,
  $$

- smoothness (2.2) between $x_{k+1}$ and $y_k$ via the descent lemma (2.3), with weight $\lambda_2=A_{k+1}$:

  $$
  f(x_{k+1})\leq f(y_k)-\tfrac{1}{2L}\|\nabla f(y_k)\|^2,
  $$

- the relation (4.15), with weight $\lambda_3=1$:

  $$
  \langle\nabla f(y_k),\,A_kx_k+a_kz_k-A_{k+1}y_k\rangle\geq0.
  $$

The weighted sum reads

$$
\begin{aligned}
0\geq{}&\lambda_1\big[f(y_k)-f(x_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle\big]\\
&+\lambda_2\big[f(x_{k+1})-f(y_k)+\tfrac{1}{2L}\|\nabla f(y_k)\|^2\big]\\
&+\lambda_3\langle\nabla f(y_k),\,A_{k+1}y_k-A_kx_k-a_kz_k\rangle ,
\end{aligned}
$$

which, using the update $z_{k+1}=z_k-\tfrac{a_k}{L}\nabla f(y_k)$ and $A_{k+1}=A_k+a_k$, together with

$$
\begin{aligned}
A_{k+1}r_{k+1}(z_{k+1})&=A_kr_k(z_{k+1})+a_k\ell_k(z_{k+1}),\\
A_kr_k(z_{k+1})&=A_kr_k(z_k)+\tfrac L2\|z_{k+1}-z_k\|^2,
\end{aligned}
$$

can be reformulated as

$$
0\geq\Psi_{k+1}-\Psi_k+\tfrac{A_{k+1}-a_k^2}{2L}\|\nabla f(y_k)\|^2 ,
$$

which is the claim as soon as $A_{k+1}=A_k+a_k\geq a_k^2$, concluding the proof. $\square$

### 2.2 SymPy verification

To verify the expression symbolically, one can define the following symbols. The minimal value $A_kr_k(z_k)$ of the current model is the free symbol `Ark_zk`, and the model is evaluated at $z_{k+1}$ through its quadratic expansion around $z_k$:

In [7]:
L = sp.symbols('L')
Ak = sp.symbols('A_k')
ak = sp.symbols('a_k')

xk = sp.symbols('x_k')
zk = sp.symbols('z_k')
yk = sp.symbols('y_k')
gyk = sp.symbols('\\nabla{}f(y_k)')
fyk = sp.symbols('f(y_k)')
fxk = sp.symbols('f(x_k)')
fxkp1 = sp.symbols('f(x_{k+1})')
Ark_zk = sp.symbols('A_k{}r_k(z_k)')

Akp1 = Ak + ak
zkp1 = zk - ak / L * gyk

ellk_zkp1 = fyk + gyk * (zkp1 - yk)           # ell_k(z_{k+1})
Ark_zkp1 = Ark_zk + L / 2 * (zkp1 - zk) ** 2  # A_k r_k(z_{k+1})
Arkp1_zkp1 = Ark_zkp1 + ak * ellk_zkp1        # A_{k+1} r_{k+1}(z_{k+1}) = A_k r_k(z_{k+1}) + a_k ell_k(z_{k+1})

Now, one can write the three inequalities used in the proof (in the form "$\ldots\leqslant 0$") and their weights to form the weighted sum:

In [8]:
inequality1 = fyk - fxk + gyk * (xk - yk)
inequality2 = fxkp1 - fyk + 1 / 2 / L * gyk ** 2
inequality3 = - gyk * (Ak * xk + ak * zk - Akp1 * yk)
lambda1 = Ak
lambda2 = Akp1
lambda3 = 1

weighted_sum = lambda1 * inequality1 + lambda2 * inequality2 + lambda3 * inequality3

Similarly, one can define the two values of the gap and form the target inequality

In [9]:
potentialk = Ak * fxk - Ark_zk
potentialkp1 = Akp1 * fxkp1 - Arkp1_zkp1

target_inequality = potentialkp1 - potentialk + (Akp1 - ak ** 2) / 2 / L * gyk ** 2

It remains to check that the expressions match, which we check by verifying their difference is identically zero

In [10]:
check(weighted_sum - target_inequality)

0

As complements, in the order of the checks below:

- the point $z_{k+1}=z_k-\tfrac{a_k}{L}\nabla f(y_k)$ minimizes $A_{k+1}r_{k+1}$;
- the relation between the gap and the potential $\Phi_k=A_k\big(f(x_k)-f_\star\big)+\tfrac L2\|z_k-x_\star\|^2$ of (2.15) stated after the proof, $\Psi_k=\Phi_k-\tfrac L2\|x_0-x_\star\|^2+\sum_{i=0}^{k-1}a_i\big(f_\star-\ell_i(x_\star)\big)$, checked on its increments: $\Psi_{k+1}-\Psi_k=\Phi_{k+1}-\Phi_k+a_k\big(f_\star-\ell_k(x_\star)\big)$ (the inequality between $x_\star$ and $y_k$ of Theorem 2.5 is delegated to the lower model);
- the increments $a_k=(1+\sqrt{4A_k+1})/2$ of Corollary 4.8 satisfy $a_k^2=A_{k+1}$;
- the initial gap (4.16): with $A_1=1$ and $x_0=y_0$, the model $r_1$ is minimized at $z_1=y_0-\tfrac1L\nabla f(y_0)$ and $\Psi_1=f(x_1)-\big(f(y_0)-\tfrac1{2L}\|\nabla f(y_0)\|^2\big)$, which is nonpositive by the descent lemma.

In [11]:
z = sp.symbols('z')
xs = sp.symbols('x_\\star')
fs = sp.symbols('f_\\star')
Arkp1_z = Ark_zk + L / 2 * (z - zk) ** 2 + ak * (fyk + gyk * (z - yk))        # A_{k+1} r_{k+1}(z)
Phik = Ak * (fxk - fs) + L / 2 * (zk - xs) ** 2                              # potential (2.15)
Phikp1 = Akp1 * (fxkp1 - fs) + L / 2 * (zkp1 - xs) ** 2
ellk_xs = fyk + gyk * (xs - yk)                                               # ell_k(x_star)
ak_fgm = (1 + sp.sqrt(4 * Ak + 1)) / 2

y0, gy0, fy0, fx1 = sp.symbols('y_0 \\nabla{}f(y_0) f(y_0) f(x_1)')
r1_z = fy0 + gy0 * (z - y0) + L / 2 * (z - y0) ** 2                           # r_1 with A_1 = 1 and x_0 = y_0
z1 = y0 - gy0 / L

(check(sp.diff(Arkp1_z, z).subs(z, zkp1)),
 check((potentialkp1 - potentialk) - (Phikp1 - Phik) - ak * (fs - ellk_xs)),
 check((ak ** 2 - Akp1).subs(ak, ak_fgm)),
 check(sp.diff(r1_z, z).subs(z, z1)),
 check((fx1 - r1_z.subs(z, z1)) - (fx1 - (fy0 - gy0 ** 2 / 2 / L))))

(0, 0, 0, 0, 0)

## 3 Verification of Theorem 4.10

### 3.1 Statement and proof (as in the manuscript)

Here $m\geq0$, the pieces are $\ell_i(x)=f(y_i)+\langle\nabla f(y_i),\,x-y_i\rangle+\tfrac m2\|x-y_i\|^2$, and the regularized model (4.17) is
$$
r_k(x)=\frac{1}{A_k}\sum_{i=0}^{k-1}a_i\,\ell_i(x)+\frac{L}{2A_k}\|x-x_0\|^2 ,
$$
so that $A_kr_k$ is a quadratic with curvature $L+mA_k$ minimized at $z_k$. Optimizing $r_{k+1}$ gives (4.18),
$$
z_{k+1}=z_k+q\delta_k\,(y_k-z_k)-\tfrac{\delta_k}{L}\nabla f(y_k),\qquad\text{with }\ \delta_k=\tfrac{a_k}{1+qA_{k+1}} .
$$
The theorem is about one iteration of the following method.

**Algorithm 4.9. (Cheap regularized model-based method, $m\geq0$)** With the increment $a_k\geq0$, the algorithm applies the following update. Given the minimizer of an upper model $x_k$ and a lower model $l_k\leq f$ with weight $A_k>0$, compute $y_k$ as a solution to (2.22),
$$
\langle\nabla f(y_k),\,A_k(x_k-y_k)+\delta_k(1+qA_k)(z_k-y_k)\rangle\geq0,
$$
with $z_k=\mathrm{argmin}\;r_k$, where $r_k(x)=l_k(x)+\tfrac{L}{2A_k}\|x-x_0\|^2$, and output a new upper model minimizer $x_{k+1}=y_k-\tfrac1L\nabla f(y_k)$ and a new lower model $l_{k+1}=(A_kl_k+a_k\ell_k)/A_{k+1}$, where $\ell_k$ is given by (4.1) at $y_k$, with the new weight $A_{k+1}=A_k+a_k$.

**Theorem 4.10. (Cheap regularized model-based method, $m\geq0$)** Let $f\in\mathcal{F}_{m,L}$. At step $k\geq1$, given $x_k$ and $l_k$, let $x_{k+1}$ and $l_{k+1}$ be generated by the cheap regularized model-based method (Algorithm 4.9). Let $\Psi_k=A_k\big(f(x_k)-r_k(z_k)\big)$ be the gap in (4.12) with weights $A_k>0$ and increments $A_{k+1}=A_k+a_k$ with $a_k\geq0$. If $a_k^2\leq A_{k+1}(1+qA_{k+1})$, then
$$
\Psi_{k+1}\leqslant\Psi_k .
$$

*Proof.* The proof consists of a weighted sum of the three inequalities of the proof of Theorem 4.7 with an additional strongly convex piece:

- convexity (2.1) between $x_k$ and $y_k$, with weight $\lambda_1=A_k$:

  $$
  f(x_k)\geq f(y_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle,
  $$

- the descent lemma (2.3), with weight $\lambda_2=A_{k+1}$:

  $$
  f(x_{k+1})\leq f(y_k)-\tfrac{1}{2L}\|\nabla f(y_k)\|^2,
  $$

- the relation (2.22), with weight $\lambda_3=1$:

  $$
  \langle\nabla f(y_k),\,A_k(x_k-y_k)+\delta_k(1+qA_k)(z_k-y_k)\rangle\geq0.
  $$

The weighted sum reads

$$
\begin{aligned}
0\geq{}&\lambda_1\big[f(y_k)-f(x_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle\big]\\
&+\lambda_2\big[f(x_{k+1})-f(y_k)+\tfrac{1}{2L}\|\nabla f(y_k)\|^2\big]\\
&+\lambda_3\langle\nabla f(y_k),\,A_k(y_k-x_k)+\delta_k(1+qA_k)(y_k-z_k)\rangle ,
\end{aligned}
$$

which, using the update (4.18) and $A_{k+1}=A_k+a_k$, together with

$$
\begin{aligned}
A_{k+1}r_{k+1}(z_{k+1})&=A_kr_k(z_{k+1})+a_k\ell_k(z_{k+1}),\\
A_kr_k(z_{k+1})&=A_kr_k(z_k)+\tfrac{L+mA_k}{2}\|z_{k+1}-z_k\|^2,
\end{aligned}
$$

as $A_kr_k$ is a quadratic with curvature $L+mA_k$ minimized at $z_k$, can be reformulated as

$$
0\geq\Psi_{k+1}-\Psi_k+\tfrac{A_{k+1}-a_k\delta_k}{2L}\|\nabla f(y_k)\|^2+\delta_k(1+qA_k)\,\tfrac m2\|y_k-z_k\|^2 ,
$$

where both coefficients are nonnegative under our assumption, as $A_{k+1}-a_k\delta_k\geq0$ is equivalent to $A_{k+1}(1+qA_{k+1})-a_k^2\geq0$, thereby concluding the proof. $\square$

### 3.2 SymPy verification

To verify the expression symbolically, one can define the following symbols:

In [12]:
L = sp.symbols('L')
m = sp.symbols('m')
Ak = sp.symbols('A_k')
ak = sp.symbols('a_k')

xk = sp.symbols('x_k')
zk = sp.symbols('z_k')
yk = sp.symbols('y_k')
gyk = sp.symbols('\\nabla{}f(y_k)')
fyk = sp.symbols('f(y_k)')
fxk = sp.symbols('f(x_k)')
fxkp1 = sp.symbols('f(x_{k+1})')
Ark_zk = sp.symbols('A_k{}r_k(z_k)')

q = m / L
Akp1 = Ak + ak
deltak = ak / (1 + q * Akp1)
zkp1 = zk + q * deltak * (yk - zk) - deltak / L * gyk        # update (4.18)

ellk_zkp1 = fyk + gyk * (zkp1 - yk) + m / 2 * (zkp1 - yk) ** 2
Ark_zkp1 = Ark_zk + (L + m * Ak) / 2 * (zkp1 - zk) ** 2      # quadratic, curvature L + m A_k, minimized at z_k
Arkp1_zkp1 = Ark_zkp1 + ak * ellk_zkp1

Now, as before, one can write the inequalities used in the proof (in the form "$\ldots\leqslant 0$") and their weights to form the weighted sum:

In [13]:
inequality1 = fyk - fxk + gyk * (xk - yk)
inequality2 = fxkp1 - fyk + 1 / 2 / L * gyk ** 2
inequality3 = gyk * (Ak * (yk - xk) + deltak * (1 + q * Ak) * (yk - zk))
lambda1 = Ak
lambda2 = Akp1
lambda3 = 1

weighted_sum = lambda1 * inequality1 + lambda2 * inequality2 + lambda3 * inequality3

Similarly, one can define the two values of the gap and form the target inequality

In [14]:
potentialk = Ak * fxk - Ark_zk
potentialkp1 = Akp1 * fxkp1 - Arkp1_zkp1

target_inequality = potentialkp1 - potentialk + (Akp1 - ak * deltak) / 2 / L * gyk ** 2 + deltak * (1 + q * Ak) * m / 2 * (yk - zk) ** 2

It remains to check that the expressions match, which we check by verifying their difference is identically zero

In [15]:
check(weighted_sum - target_inequality)

0

As complements, in the order of the checks below:

- the point $z_{k+1}$ of (4.18) minimizes $A_{k+1}r_{k+1}$;
- the coefficient of the gradient term: $A_{k+1}-a_k\delta_k=\tfrac{A_{k+1}(1+qA_{k+1})-a_k^2}{1+qA_{k+1}}$;
- the increments (2.23) of Corollary 4.11 satisfy $a_k^2=A_{k+1}(1+qA_{k+1})$, and give the initial weight $A_1=\tfrac{1}{1-q}$ from $A_0=0$;
- the initial gap (4.19): with $A_1=\tfrac1{1-q}$ and $x_0=y_0$, the curvature of $r_1$ is $m+L/A_1=L$, its minimizer is $z_1=y_0-\tfrac1L\nabla f(y_0)$, and $r_1(z_1)=f(y_0)-\tfrac1{2L}\|\nabla f(y_0)\|^2$, so that $\Psi_1\leq0$ by the descent lemma.

In [16]:
z = sp.symbols('z')
Arkp1_z = Ark_zk + (L + m * Ak) / 2 * (z - zk) ** 2 + ak * (fyk + gyk * (z - yk) + m / 2 * (z - yk) ** 2)   # A_{k+1} r_{k+1}(z)
ak_fgm = (1 + 2 * q * Ak + sp.sqrt(1 + 4 * Ak * (1 + q * Ak))) / 2 / (1 - q)                           # increment (2.23)

y0, gy0, fy0 = sp.symbols('y_0 \\nabla{}f(y_0) f(y_0)')
A1 = 1 / (1 - q)
r1_z = fy0 + gy0 * (z - y0) + m / 2 * (z - y0) ** 2 + L / 2 / A1 * (z - y0) ** 2   # r_1, with x_0 = y_0
z1 = y0 - gy0 / L

(check(sp.diff(Arkp1_z, z).subs(z, zkp1)),
 check((Akp1 - ak * deltak) - (Akp1 * (1 + q * Akp1) - ak ** 2) / (1 + q * Akp1)),
 check((ak ** 2 - Akp1 * (1 + q * Akp1)).subs(ak, ak_fgm)),
 check(ak_fgm.subs(Ak, 0) - A1),
 check(m + L / A1 - L),
 check(sp.diff(r1_z, z).subs(z, z1)),
 check(r1_z.subs(z, z1) - (fy0 - gy0 ** 2 / 2 / L)))

(0, 0, 0, 0, 0, 0, 0)

## 4 Verification of the subproblems of Section 4.5

Section 4.5 (better lower models) replaces the averaged lower model by a better one, $\tilde l_{k+1}$, and converts the room that it leaves into a larger weight by solving the one-dimensional problem (4.22),
$$
A_{k+1}^\star=\max\big\{A:\ f(x_{k+1})\leq\tilde v_{k+1}(A)\big\},\qquad\text{with }\ \tilde v_{k+1}(A)=\min_{x}\Big\{\tilde l_{k+1}(x)+\tfrac{L}{2A}\|x-x_0\|^2\Big\}.
$$
The manuscript gives the subproblems to be solved in explicit form. Evaluating $\tilde v_{k+1}(A)$ is a quadratic program over the simplex: $\max_\lambda v_\lambda$ for quadratic averaging with memory, (4.25) for the piecewise-linear model, and (4.29) for the model with one aggregate. With one aggregate, the weight problem (4.22) itself becomes the maximization of the ratio (4.31), solved by (4.32), or by (4.34) without regularization. This part verifies the algebra leading to these expressions.

Exchanging the minimum over $x$ and the maximum over $\lambda$ (Sion's minimax theorem) is not algebra, and is not checked here: what is checked is every explicit formula obtained after the exchange.

### 4.1 Minimizing a regularized quadratic: (4.29)

All the models of Section 4.5 are maxima of convex combinations of the pieces (4.1). Any such combination $w$ is a quadratic with curvature $m$, which we write around $x_0$ as
$$
w(z)=w(x_0)+\langle\bar g,\,z-x_0\rangle+\tfrac m2\|z-x_0\|^2,\qquad\text{with }\ \bar g=\nabla w(x_0).
$$
With the parameter $\gamma\in[0,1]$ of (4.28) in the regularization, the claim behind (4.29) is
$$
\min_{z}\Big\{w(z)+\tfrac{\gamma L}{2A}\|z-x_0\|^2\Big\}=w(x_0)-\tfrac{t}{2L}\|\bar g\|^2,\qquad\text{attained at }\ z=x_0-\tfrac tL\,\bar g,\qquad\text{with }\ t=\tfrac{A}{\gamma+qA}.
$$
The checks below are, in this order: the claimed point is a stationary point of the objective (a convex quadratic); the minimal value; the two particular cases $t=A$ for $\gamma=1$ and $m=0$, as in (4.25), and $t=\tfrac1q$ for $\gamma=0$, where the weight $A$ disappears; the inverse map $A=\tfrac{\gamma t}{1-qt}$ used in (4.31); and the derivative of the minimal value with respect to the weight, which is $-\tfrac{\gamma L}{2A^2}\|z-x_0\|^2\leq0$ at the minimizer $z$. The minimal value is thus nonincreasing in $A$, as used before (4.21), and this derivative is the one needed by Newton's method on $\phi(A)=f(x_{k+1})-\tilde v_{k+1}(A)$ (at the maximizing weights $\lambda$, by the envelope theorem).

In [17]:
L = sp.symbols('L')
m = sp.symbols('m')
A = sp.symbols('A')
gamma = sp.symbols('gamma')

x0 = sp.symbols('x_0')
z = sp.symbols('z')
w_x0 = sp.symbols('w(x_0)')
gbar = sp.symbols('\\bar{g}')

q = m / L
t = A / (gamma + q * A)

w_z = w_x0 + gbar * (z - x0) + m / 2 * (z - x0) ** 2          # a quadratic with curvature m, written around x_0
objective = w_z + gamma * L / 2 / A * (z - x0) ** 2            # inner minimization of (4.28)
z_min = x0 - t / L * gbar                                     # claimed minimizer
v_min = w_x0 - t / 2 / L * gbar ** 2                           # claimed minimal value, as in (4.29)

(check(sp.diff(objective, z).subs(z, z_min)),
 check(objective.subs(z, z_min) - v_min),
 check(t.subs({gamma: 1, m: 0}) - A),
 check(t.subs(gamma, 0) - 1 / q),
 check(gamma * t / (1 - q * t) - A),
 check(sp.diff(v_min, A) + gamma * L / 2 / A ** 2 * (z_min - x0) ** 2))

(0, 0, 0, 0, 0, 0)

### 4.2 Convex combinations of the pieces: $v_\lambda$ and (4.25)

For weights $\lambda$ in the simplex, the combination $l_\lambda=\sum_i\lambda_i\ell_i$ of the pieces (4.1) is a quadratic of the above form, with
$$
l_\lambda(x_0)=\sum_i\lambda_i\,\ell_i(x_0),\qquad \bar g_\lambda=\nabla l_\lambda(x_0)=\sum_i\lambda_i\big(\nabla f(y_i)+m\,(x_0-y_i)\big),
$$
that is, $\bar g_\lambda=\sum_i\lambda_i\nabla f(y_i)$ when $m=0$. The checks below use three pieces with weights $\lambda_0$, $\lambda_1$ and $1-\lambda_0-\lambda_1$. In this order:

- the gradient of $l_\lambda$ at $x_0$ and the expansion of $l_\lambda$ around $x_0$, so that part 4.1 applies to it;
- quadratic averaging with memory (no regularization, $m>0$): the minimal value $l_\lambda(x_0)-\tfrac1{2m}\|\bar g_\lambda\|^2$ of $l_\lambda$ is the value $v_\lambda$ of the manuscript,
$$
v_\lambda=\sum_i\lambda_i\Big(f(y_i)-\tfrac1{2m}\|\nabla f(y_i)\|^2+\tfrac m2\|w_i-x_0\|^2\Big)-\tfrac m2\Big\|\sum_i\lambda_i(w_i-x_0)\Big\|^2,\qquad\text{with }\ w_i=y_i-\tfrac1m\nabla f(y_i),
$$
which is also the value $v_k$ of (4.4), and the minimizer $x_0-\tfrac1m\bar g_\lambda$ is $\sum_i\lambda_iw_i$;
- the piecewise-linear model ($\gamma=1$, $m=0$): the minimizer $x_0-\tfrac AL\bar g_\lambda$ and the minimal value $l_\lambda(x_0)-\tfrac{A}{2L}\|\bar g_\lambda\|^2$ of $l_\lambda+\tfrac{L}{2A}\|\cdot-x_0\|^2$, which is the function maximized in (4.25).

Both $v_\lambda$ and the function maximized in (4.25) are a linear function of $\lambda$ minus the squared norm of a linear function of $\lambda$: they are concave quadratics, hence the quadratic programs over the simplex.

In [18]:
y0, y1, y2 = sp.symbols('y_0 y_1 y_2')
gy0, gy1, gy2 = sp.symbols('\\nabla{}f(y_0) \\nabla{}f(y_1) \\nabla{}f(y_2)')
fy0, fy1, fy2 = sp.symbols('f(y_0) f(y_1) f(y_2)')
lam0, lam1 = sp.symbols('lambda_0 lambda_1')
pieces = [(lam0, y0, gy0, fy0), (lam1, y1, gy1, fy1), (1 - lam0 - lam1, y2, gy2, fy2)]

def ell(y, g, f, x):
    return f + g * (x - y) + m / 2 * (x - y) ** 2             # piece (4.1) built at y, evaluated at x

l_z = sum(lam * ell(y, g, f, z) for (lam, y, g, f) in pieces)                    # l_lambda(z)
l_x0 = sum(lam * ell(y, g, f, x0) for (lam, y, g, f) in pieces)                  # l_lambda(x_0)
gbar_lam = sum(lam * (g + m * (x0 - y)) for (lam, y, g, f) in pieces)            # claimed gradient of l_lambda at x_0

# quadratic averaging with memory
w_pts = [y - g / m for (lam, y, g, f) in pieces]                                 # w_i = y_i - grad f(y_i) / m
v_lam = (sum(lam * (f - g ** 2 / 2 / m + m / 2 * (w - x0) ** 2) for ((lam, y, g, f), w) in zip(pieces, w_pts))
         - m / 2 * sum(lam * (w - x0) for ((lam, y, g, f), w) in zip(pieces, w_pts)) ** 2)   # v_lambda of the manuscript
z_lam = sum(lam * w for ((lam, y, g, f), w) in zip(pieces, w_pts))                # sum of lambda_i w_i
v_44 = sum(lam * (f - g ** 2 / 2 / m + m / 2 * (w - z_lam) ** 2) for ((lam, y, g, f), w) in zip(pieces, w_pts))   # v_k of (4.4)

# piecewise-linear model: m = 0 and regularization L/(2A)
l_z_affine, l_x0_affine, gbar_affine = l_z.subs(m, 0), l_x0.subs(m, 0), gbar_lam.subs(m, 0)
objective_affine = l_z_affine + L / 2 / A * (z - x0) ** 2
z_A = x0 - A / L * gbar_affine

(check(sp.diff(l_z, z).subs(z, x0) - gbar_lam),
 check(l_z - (l_x0 + gbar_lam * (z - x0) + m / 2 * (z - x0) ** 2)),
 check((l_x0 - gbar_lam ** 2 / 2 / m) - v_lam),
 check(v_lam - v_44),
 check((x0 - gbar_lam / m) - z_lam),
 check(sp.diff(objective_affine, z).subs(z, z_A)),
 check(objective_affine.subs(z, z_A) - (l_x0_affine - A / 2 / L * gbar_affine ** 2)))

(0, 0, 0, 0, 0, 0, 0)

### 4.3 One aggregate and one new piece: (4.30), (4.31), (4.32) and (4.34)

With a single aggregate $w_k$ and the new piece $\ell_k$, the combinations are $w_\lambda=(1-\lambda)\,w_k+\lambda\,\ell_k$ with $\lambda\in[0,1]$. The values are measured relative to $f(x_{k+1})$ as in (4.30): $e_\lambda=w_\lambda(x_0)-f(x_{k+1})=(1-\lambda)\,e_0+\lambda\,e_1$ and $\bar g_\lambda=(1-\lambda)\,\bar g_0+\lambda\,\bar g_1$. By part 4.1, the certificate (4.21) then reads $e_\lambda-\tfrac{t}{2L}\|\bar g_\lambda\|^2\geq0$ for some $\lambda\in[0,1]$.

The expressions below involve ratios of inner products of $\bar g_0$ and $\bar g_1$. We therefore work with the three scalars $G_{00}=\|\bar g_0\|^2$, $G_{01}=\langle\bar g_0,\bar g_1\rangle$ and $G_{11}=\|\bar g_1\|^2$, which makes the checks valid in any dimension:
$$
\|\bar g_\lambda\|^2=(1-\lambda)^2G_{00}+2\lambda(1-\lambda)G_{01}+\lambda^2G_{11},\quad
\|\bar g_1-\bar g_0\|^2=G_{00}-2G_{01}+G_{11},\quad
\|e_0\bar g_1-e_1\bar g_0\|^2=e_0^2G_{11}-2e_0e_1G_{01}+e_1^2G_{00}.
$$
The checks are, in this order:

- (4.30): the gradient at $x_0$ of the new piece is $\bar g_1=\nabla f(y_k)+m\,(x_0-y_k)$;
- (4.31): for a given $\lambda$, the certificate is linear in $t$ and vanishes at $t=2L\,e_\lambda/\|\bar g_\lambda\|^2$, which is thus the largest value of $t$ certified by $\lambda$; and $A=\tfrac{\gamma t}{1-qt}$ is increasing in $t$, so that the largest weight corresponds to the largest ratio;
- (4.32): the derivative of the ratio $e_\lambda/\|\bar g_\lambda\|^2$ with respect to $\lambda$ is
$$
-\frac{(e_1-e_0)\,\|\bar g_1-\bar g_0\|^2\,(\lambda-\lambda_+)(\lambda-\lambda_-)}{\|\bar g_\lambda\|^4},\qquad\text{with }\ \lambda_\pm=\frac{1}{e_1-e_0}\bigg(-e_0\pm\frac{\|e_0\,\bar g_1-e_1\,\bar g_0\|}{\|\bar g_1-\bar g_0\|}\bigg),
$$
so that the stationary points of the ratio are the two points of (4.32). To avoid square roots, the product is written as $(\lambda-\lambda_+)(\lambda-\lambda_-)=\big(\lambda+\tfrac{e_0}{e_1-e_0}\big)^2-\tfrac{\|e_0\bar g_1-e_1\bar g_0\|^2}{(e_1-e_0)^2\,\|\bar g_1-\bar g_0\|^2}$;
- (4.34): without regularization ($\gamma=0$, $t=\tfrac1q$), the function $e_\lambda-\tfrac1{2m}\|\bar g_\lambda\|^2$ to be maximized is a concave quadratic in $\lambda$, with second derivative $-\tfrac1m\|\bar g_1-\bar g_0\|^2$, and its stationary point is the argument of the projection in (4.34), $\big(m\,(e_1-e_0)-\langle\bar g_0,\,\bar g_1-\bar g_0\rangle\big)/\|\bar g_1-\bar g_0\|^2$.

In [19]:
e0, e1 = sp.symbols('e_0 e_1')
G00, G01, G11 = sp.symbols('G_{00} G_{01} G_{11}')
lam = sp.symbols('lambda')
tt = sp.symbols('t')

yk, gyk, fyk = sp.symbols('y_k \\nabla{}f(y_k) f(y_k)')

e_lam = (1 - lam) * e0 + lam * e1
G_lam = (1 - lam) ** 2 * G00 + 2 * lam * (1 - lam) * G01 + lam ** 2 * G11       # |g_lambda|^2
D = G00 - 2 * G01 + G11                                                          # |g_1 - g_0|^2
N = e0 ** 2 * G11 - 2 * e0 * e1 * G01 + e1 ** 2 * G00                            # |e_0 g_1 - e_1 g_0|^2

certificate = e_lam - tt / 2 / L * G_lam
ratio = e_lam / G_lam
product = (lam + e0 / (e1 - e0)) ** 2 - N / (e1 - e0) ** 2 / D                   # (lambda - lambda_+)(lambda - lambda_-)

objective_unregularized = e_lam - G_lam / 2 / m
lam_oqa = (m * (e1 - e0) - (G01 - G00)) / D                                      # argument of the projection in (4.34)

(check(sp.diff(ell(yk, gyk, fyk, z), z).subs(z, x0) - (gyk + m * (x0 - yk))),
 check(certificate.subs(tt, 2 * L * ratio)),
 check(sp.diff(gamma * tt / (1 - q * tt), tt) - gamma / (1 - q * tt) ** 2),
 check(sp.diff(ratio, lam) + (e1 - e0) * D * product / G_lam ** 2),
 check(sp.diff(objective_unregularized, lam, 2) + D / m),
 check(sp.diff(objective_unregularized, lam).subs(lam, lam_oqa)))

(0, 0, 0, 0, 0, 0)

### 4.4 The a priori weight is feasible, and the first iteration of Algorithm 4.12

The manuscript states that the a priori weight $A_{k+1}=\tilde A_k+a_k$ is feasible for (4.22), as it corresponds to $\lambda=a_k/A_{k+1}$ in (4.31). Indeed, Theorem 4.10 only uses the lower model through the curvature of $A_kr_k$, its minimizer $z_k$ and its minimal value (the free symbol `Ark_zk` of part 3), so that it applies to the aggregate $w_k$ with the weight $\tilde A_k$. What remains to be checked is that the quantities of Theorem 4.10 are those of (4.29) for these weights. With $t_k=\tfrac{\tilde A_k}{1+q\tilde A_k}$ and $t=\tfrac{A_{k+1}}{1+qA_{k+1}}$ (that is, $\gamma=1$), the first two checks are:

- starting from $z_k=x_0-\tfrac{t_k}{L}\,\bar g_0$, the update (4.18) gives $z_{k+1}=x_0-\tfrac tL\,\bar g_\lambda$ with $\lambda=a_k/A_{k+1}$;
- the value $A_{k+1}r_{k+1}(z_{k+1})$ of the proof of Theorem 4.10 is $A_{k+1}\big(w_\lambda(x_0)-\tfrac{t}{2L}\|\bar g_\lambda\|^2\big)$ for the same $\lambda$.

Hence $\Psi_{k+1}\leq\Psi_k\leq0$ reads $e_\lambda-\tfrac{t}{2L}\|\bar g_\lambda\|^2\geq0$ for these $\lambda$ and $t$: the solution of (4.31) satisfies $t_{k+1}\geq t$, that is, $\tilde A_{k+1}\geq A_{k+1}$.

The last two checks are about the first iteration of Algorithm 4.12 (convention $\lambda_0=1$, with $y_0=x_0$). Writing $f(x_1)=f(x_0)-\tfrac1{2L}\|\nabla f(x_0)\|^2-s$, where $s\geq0$ by the descent lemma, we get $t_1=2L\,e_1/\|\bar g_1\|^2=1+2Ls/\|\nabla f(x_0)\|^2\geq1$, and $\tfrac{t_1}{1-qt_1}-\tfrac1{1-q}=\tfrac{t_1-1}{(1-qt_1)(1-q)}\geq0$: the first realized weight $\tilde A_1$ is at least the initial weight $A_1=\tfrac1{1-q}$ of Algorithm 4.9.

In [20]:
At, ak = sp.symbols('\\tilde{A}_k a_k')
wk_x0, g0bar = sp.symbols('w_k(x_0) \\bar{g}_0')

Akp1 = At + ak
deltak = ak / (1 + q * Akp1)
t_k = At / (1 + q * At)                                        # t for the weight A~_k
t_prior = Akp1 / (1 + q * Akp1)                                # t for the a priori weight A_{k+1}
zk = x0 - t_k / L * g0bar                                      # minimizer of r_k = w_k + L/(2 A~_k) |. - x_0|^2, by 4.1
rk_zk = wk_x0 - t_k / 2 / L * g0bar ** 2                       # r_k(z_k), by 4.1

lam_prior = ak / Akp1
g1bar = gyk + m * (x0 - yk)
g_lam = (1 - lam_prior) * g0bar + lam_prior * g1bar
w_lam_x0 = (1 - lam_prior) * wk_x0 + lam_prior * ell(yk, gyk, fyk, x0)

zkp1 = zk + q * deltak * (yk - zk) - deltak / L * gyk          # update (4.18)
Arkp1_zkp1 = At * rk_zk + (L + m * At) / 2 * (zkp1 - zk) ** 2 + ak * ell(yk, gyk, fyk, zkp1)   # as in the proof of Theorem 4.10

fx0, gx0, s = sp.symbols('f(x_0) \\nabla{}f(x_0) s')
fx1 = fx0 - gx0 ** 2 / 2 / L - s
t1 = 2 * L * (fx0 - fx1) / gx0 ** 2                            # e_1 = ell_0(x_0) - f(x_1) = f(x_0) - f(x_1), g_1 = grad f(x_0)

(check(zkp1 - (x0 - t_prior / L * g_lam)),
 check(Arkp1_zkp1 - Akp1 * (w_lam_x0 - t_prior / 2 / L * g_lam ** 2)),
 check(t1 - (1 + 2 * L * s / gx0 ** 2)),
 check(tt / (1 - q * tt) - 1 / (1 - q) - (tt - 1) / (1 - q * tt) / (1 - q)))

(0, 0, 0, 0)